In [ ]:
import numpy as np
import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchmetrics
import kornia.augmentation as K
import pandas as pd
from statistics import median

In [6]:
PROJECT_ROOT = Path().resolve().parent
datetime_now = datetime.datetime.now()
date = datetime_now.strftime("%Y-%m-%d")

loss_curve_fp = PROJECT_ROOT / "outputs/plots/loss_curves" / f"loss_curve_{date}.png"

SEED= 42
torch.manual_seed(SEED)

In [34]:
patches_path = PROJECT_ROOT / "outputs/15px_patches" / "combined_15px_patches.geojson"

with open(patches_path, "r") as f:
    patches = json.load(f)


In [35]:
# Version using comprehension

def load_patch_dict_as_tensor(Patch_dict, band_list = None):
    if not band_list:
        band_list = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

    feature_list = Patch_dict["features"]

    patches = np.stack([
        np.stack([np.array(F["properties"][band]) for band in band_list]) # (N, C, H, W) which is what pytorch expects
        for F in feature_list  
    ])

    labels = torch.tensor([F["properties"]["lc"] for F in feature_list]).float().unsqueeze(1) # BCElogits loss will need to compare labels to float logits or probabilties
    # .unsqueeze(1) changes the shape from ([0,1,1,0]) to ([0],[1],[1],[0]) aka (B,) to (B, 1), need this for the criterion. 

    locations = np.array([F["properties"]["location"] for F in feature_list]) # array for easier boolean masking but no need for tensor

    class_int = np.array([F["properties"]["class_int"] for F in feature_list])
    
    label_id = np.array([F["properties"]["label_id"] for F in feature_list])

    patches = torch.from_numpy(patches).float()


    return patches, labels, locations, class_int, label_id



# check the shape

#print(patches.shape)


In [9]:
print(labels.shape)
print(label_id[12000])

NameError: name 'labels' is not defined

In [42]:
# next need to calcualte the mean and std for just the train region, can create a test_mask and val_mask and then train_masl = ~test_mask & ~val_mask

test_region = "Winam"
val_region = "Rodman"

test_mask = locations == test_region
val_mask = locations == val_region

train_mask = ~test_mask & ~ val_mask

train_patches, train_labels = patches[train_mask], labels[train_mask]
val_patches, val_labels = patches[val_mask], labels[val_mask]
test_patches, test_labels, test_labels_ids = patches[test_mask], labels[test_mask], label_id[test_mask]


mean = train_patches.mean(dim= (0,2,3)) # will average over the 0,2,3 axes and keep the 1 axis (channels / bands) separate. output shape (C,)
std = train_patches.std(dim = (0,2,3))

# mean = torch.tensor(mean).view(1,-1,1,1) # changes mean from (C,) to (1,C,1,1)
# std = torch.tensor(std).view(1,-1,1,1)

# standardise all the patches using the mean and std from the train patches - np maybe do this for each patch in the dataset in __getitem__?

# std_train_patches = (train_patches - mean)/std
# std_val_patches = (val_patches-mean)/std
# std_test_patches = (test_patches-mean)/ std




In [13]:
print(val_labels.shape)
print(test_labels_ids.shape)
print(type(test_labels_ids))
print(test_patches.shape)

torch.Size([2072, 1])
(0,)
<class 'numpy.ndarray'>
torch.Size([0, 10, 31, 31])


In [ ]:
### Might need to unsqueeze label ids and cals int. What shape wil the predicitons be? These three are all np.,arrays at this point (B,)

In [43]:

class PatchDataset(Dataset):
    def __init__(self, patches, labels, mean, std):
        
        mean = mean.detach().clone().view(1,-1,1,1) # changes from (C,) to (1,C,1,1)
        std = std.detach().clone().view(1,-1,1,1)
    
        self.patches = (patches - mean)/std 
        self.labels = labels 

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        return self.patches[idx], self.labels[idx]
    


    
train_dataset = PatchDataset(patches = train_patches, labels = train_labels, mean = mean, std = std)

train_loader = DataLoader(
    dataset= train_dataset,
    shuffle= True,
    batch_size= 32,
    num_workers= 0,
    pin_memory= True
    )
    
val_dataset = PatchDataset(patches = val_patches,labels= val_labels, mean = mean, std = std)

val_loader = DataLoader(
    dataset= val_dataset,
    shuffle= False,
    batch_size= 32,
    num_workers= 0,
    pin_memory= True
    )

# # --- Device selection ---
# # Colab (with a GPU runtime: Runtime > Change runtime type > GPU):
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Apple Silicon (M1/M2/M3/M4), using the Metal Performance Shaders backend:
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Portable version that works on either without editing — checks cuda, then mps, then falls back to cpu:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

aug = K.AugmentationSequential(
    K.RandomRotation(degrees = 90.0, p =0.5),
    K.RandomHorizontalFlip(p= 0.25),
    K.RandomVerticalFlip(p= 0.25),
    data_keys= ['input']).to(device)

# for epoch in len(number_epochs):
#     # Training
#     #-----------------------
#     model.train()

#     for batch_X, batch_y in train_loader:
#         batch_X = batch_X.to(device)
#         batch_y = batch_y.to(device)

#         batch_X = aug(batch_X) # Augment only in the train split

#         optimizer.zero_grad()
#         output = model(batch_x)
#         loss = criterion(output, batch_y)
#         loss.backwards()
#         optimizer.step()

#     # Val
#     #-----------------------

#     model.eval()

#     with torch.no_grad():
#         for batch_X, batch_y in val_loader:
            
#             batch_X = batch_X.to(device)
#             batch_y = batch_y.to(device)

#             output = model(batch_x)
#             val_loss = criterion(output, batch_y)

    # Plot?

In [16]:
# Define my model



def buildCNN_2xVGG(input_channels= 10, dropout = 0.3):

        myCNN = nn.Sequential(
        nn.Conv2d(in_channels=input_channels, 
                out_channels = 32, 
                kernel_size=3,
                stride= 1,
                padding=1  ),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.Conv2d(in_channels=32, 
                out_channels = 32, 
                kernel_size=3,
                stride= 1,
                padding=1  ),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
                nn.Conv2d(in_channels=32, 
                out_channels = 64, 
                kernel_size=3,
                stride= 1,
                padding=1  ),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(in_channels=64, 
                out_channels = 64, 
                kernel_size=3,
                stride= 1,
                padding=1  ),
        nn.BatchNorm2d(64),
        
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Dropout(dropout),
        nn.Linear(64,1))

        return myCNN

In [ ]:
# metrics - use torchmetrics to make a metric collection that is a dictionary of metrics

metrics = torchmetrics.MetricCollection({
    "accuracy": torchmetrics.classification.BinaryAccuracy(),
    "f1": torchmetrics.classification.BinaryF1Score(),
    "precision": torchmetrics.classification.BinaryPrecision(),
    "recall": torchmetrics.classification.BinaryRecall(),
    "auroc": torchmetrics.classification.BinaryAUROC()
}).to(device)

In [ ]:
def train_for_one_epoch(model, train_loader, device, aug, optimizer, criterion):
    
    model.train()
    running_loss = 0.0
    metrics.reset()
    
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        batch_X = aug(batch_X) # Augment only in the train split

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        metrics.update(logits, batch_y) # accumulates the metrics per batch. Torchmetrics detects logits and applies sigmoid internally
        running_loss += loss.item()*batch_X.size(0)
    avg_loss = running_loss / len(train_loader.dataset) 

    results = {k: v.item() for k, v in metrics.compute().items()}   

    return avg_loss, results

    
def val_for_one_epoch(model, val_loader, metrics, device, criterion):
    
    model.eval()
    running_val_loss = 0.0
    metrics.reset() # clears the metrics from previous epochs
    

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X)               # Outputting logits as useing BCEntropy with logit loss as criterion
            val_loss = criterion(logits, batch_y)
            
            #probs = torch.sigmoid(logits) # gives probabilities between 0 and 1
            #preds = (logits >= 0).int() #because sigmoid(0) = 0.5 and would use as probability threshold anyway
            metrics.update(logits, batch_y) # accumulates the metrics per batch. Torchmetrics detects logits and applies sigmoid internally

            running_val_loss += val_loss.item()*batch_X.size(0)
           

        avg_loss = running_val_loss / len(val_loader.dataset)
        results = {k: v.item() for k, v in metrics.compute().items()}
        
    return avg_loss, results

def predict_on_test_region(model, test_loader, metrics, device, criterion):
    
    # no per epoch as not training, testing the model once on the held out region
    model.eval()
    metrics.reset()
    running_test_loss = 0.0
    preds_tensor = torch.empty((0,1))
    logits_tensor = torch.empty((0,1))

    with torch.no_grad():
        for batch_X, batch_y in test_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X)
            logits_tensor = torch.cat([logits_tensor, logits], dim = 0)

            test_loss = criterion(logits, batch_y)

            #probs = torch.sigmoid(logits) # gives probabilities between 0 and 1
            preds = (logits >= 0).int #because sigmoid(0) = 0.5 and would use as probability threshold anyway
            preds_tensor = torch.cat([preds_tensor, preds], dim= 0) # concat the preds tensors for each batch

            metrics.update(logits, batch_y)
            running_test_loss += test_loss.item()*batch_X.size(0)
    
    average_loss = running_test_loss / len(test_loader.dataset)

    results = {k: v.item() for k, v in metrics.compute().items()}

    return average_loss, results, preds_tensor, logits

def plot_and_save_loss(history_dict, loss_curve_fp, run_id):
    
    plt.figure(figsize=(8, 5))
    plt.plot(history_dict["train_loss"], label="Train Loss")
    if "val_loss" in history_dict.keys():
        plt.plot(history_dict["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    if "val_loss" in history_dict.keys():
        plt.title(f"Training vs Validation Loss {run_id}")
    else:
        plt.title(f"Training Loss {run_id}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(Path(loss_curve_fp), dpi=150)
    plt.show()

    

In [36]:
patches, labels, locations, class_int, label_id = load_patch_dict_as_tensor(patches, band_list= None)

In [ ]:

model = buildCNN_2xVGG(input_channels= 10, dropout = 0.3)
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

num_epochs = 50
best_val_loss = float("inf")
patience = 10
epochs_no_improve = 0
best_state = None
best_val_metrics = None
best_val_preds = None
best_epoch = None
history_dict = {"train_loss": [], "val_loss": [], "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[],
                "train_accuracy":[],"train_f1":[], "train_precision":[], "train_recall":[], "train_auroc":[]
                }

list_of_run_history = []

best_state_dict= {"run_ID": [], "test_region": [], "val_region": [], "train_loss": [], "val_loss": [], 
                  "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[], }

# ----- loop for outer fold goes here 
# loop for inner fold goes here
run_ID = None


for epoch in range(num_epochs):
    
    train_loss, train_metrics = train_for_one_epoch(model, train_loader, device, aug, optimizer, criterion)
    val_loss, val_metrics, val_preds = val_for_one_epoch(model, val_loader, metrics, device, criterion)

    history_dict["train_loss"].append(train_loss)
    history_dict["val_loss"].append(val_loss)
    history_dict["val_accuracy"].append(val_metrics["accuracy"])
    history_dict["val_f1"].append(val_metrics["f1"])
    history_dict["val_precision"].append(val_metrics["precision"])
    history_dict["val_recall"].append(val_metrics["recall"])
    history_dict["val_auroc"].append(val_metrics["auroc"])
    history_dict["train_accuracy"].append(train_metrics["accuracy"])
    history_dict["train_f1"].append(train_metrics["f1"])
    history_dict["train_precision"].append(train_metrics["precision"])
    history_dict["train_recall"].append(train_metrics["recall"])
    history_dict["train_auroc"].append(train_metrics["auroc"])

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/100 | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} "
          f"| val_f1: {val_metrics['f1']:.4f} | val_acc: {val_metrics['accuracy']:.4f}")
    
    plot_and_save_loss(history_dict, loss_curve_fp, run_id="test of 31px")

    # Early stopping

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_train_loss = train_loss
        best_val_metrics = val_metrics
        best_val_preds = val_preds
        best_epoch = epoch
        epochs_no_improve = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve +=1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epcoch: {epoch+1}")
            break

model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch+1} | val_loss: {best_val_loss:.4f} | val_metrics: {best_val_metrics}")

run_history_dict = {run_ID: history_dict}
list_of_run_history.append(run_history_dict)

best_state_dict["run_ID"].append(run_ID)
best_state_dict["test_region"].append(test_region)
best_state_dict["val_region"].append(val_region)
best_state_dict["train_loss"].append(train_loss)
best_state_dict["val_loss"].append(best_val_loss)
best_state_dict["val_accuracy"].append(best_val_metrics["accuracy"])
best_state_dict["val_f1"].append(best_val_metrics["f1"])
best_state_dict["val_precision"].append(best_val_metrics["precision"])
best_state_dict["val_recall"].append(best_val_metrics["recall"])
best_state_dict["val_auroc"].append(best_val_metrics["auroc"])




In [90]:
print(len(val_loader.dataset))
print(best_val_preds.shape)
print(type(best_val_preds))
print(history_dict["train_accuracy"])
print(best_state_dict["val_accuracy"])
print(best_state_dict["run_ID"])


2002
torch.Size([2002, 1])
<class 'torch.Tensor'>
[0.9678933024406433, 0.9714332818984985]
[0.9605394601821899]
[None]


In [ ]:
# Create a nested loop without doing hyperparameter tuning

band_list = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

patches, labels, locations, class_int, label_id = load_patch_dict_as_tensor(patches, band_list= band_list)

input_channels = len(band_list)

num_epochs = 2
best_val_loss = float("inf")
patience = 10
epochs_no_improve = 0
best_state = None
best_val_metrics = None
best_epoch = None


list_of_run_histories = [] # will store per-epoch training and validation metrics of EVERY run for later plotting
list_of_predicition_dfs = [] # will store predictions and labels for each of the outer fold test regions for confusion matrix
list_of_inner_best_state_metrics = [] # will store the training vand validation losses and other metrics for the best model for each inner fold. 

outer_metrics_dict= {"run_ID": [], "test_region": [],"epochs_trained": [], "test_loss":[], "val_region": [], "train_loss": [], "val_loss": [], 
                  "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[], }
# will store final epoch training loss, and testing loss and other metrics for the outer folds

location_list = ["Hartbeespoort", "Rodman", "Mula", "Inle", "Vembanad", "Valsequillo", "RawaPening", "Winam"]

for location in location_list:
    
    test_region = location
    remaining_regions = [loc for loc in location_list if loc != location]
    test_abrv = test_region[0:3]
    # print(test_abrv)

    # Need a fresh best_state_dict for each outer fold so can get final metrics of the inner fold runs. 
    inner_best_state_dict= {"run_ID": [], "test_region": [], "val_region": [], "train_loss": [], "val_loss": [], "epoch_stopped": [], 
                  "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[], }
    
    for region in remaining_regions:

        val_region = region
        training_regions = [loc for loc in remaining_regions if loc != region]
        val_abrv = val_region[0:3]

        run_ID = date + "_" + f"T:{test_abrv}" + "_" +f"V:{val_abrv}"

        model = buildCNN_2xVGG(input_channels= input_channels, dropout = 0.3)
        model = model.to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

        num_epochs = 2
        best_val_loss = float("inf")
        patience = 10
        epochs_no_improve = 0
        best_state = None
        best_val_metrics = None
        best_epoch = None

        # Create history_dict for each run, with values for each epoch, that will then be contanated into a long dataframe for plotting loss_curves / analysis
        history_dict = {"run_ID": [], "loop":[], "epoch":[], "train_loss": [], "val_loss": [], "val_accuracy": [], "val_f1": [], "val_precision": [], "val_recall": [], "val_auroc":[],
                        "train_accuracy":[],"train_f1":[], "train_precision":[], "train_recall":[], "train_auroc":[]
                        }

        # train_data, val_data, train laoder, val loader
        #-------------------------------------------------------
        test_mask = locations == test_region
        val_mask = locations == val_region

        train_mask = ~test_mask & ~ val_mask

        train_patches, train_labels = patches[train_mask], labels[train_mask]
        val_patches, val_labels = patches[val_mask], labels[val_mask]
        # test_patches, test_labels, test_labels_ids = patches[test_mask], labels[test_mask], label_id[test_mask]


        mean = train_patches.mean(dim= (0,2,3)) # will average over the 0,2,3 axes and keep the 1 axis (channels / bands) separate. output shape (C,)
        std = train_patches.std(dim = (0,2,3))

        train_dataset = PatchDataset(patches = train_patches, labels = train_labels, mean = mean, std = std)

        train_loader = DataLoader(
            dataset= train_dataset,
            shuffle= True,
            batch_size= 32,
            num_workers= 0,
            pin_memory= True
            )
            
        val_dataset = PatchDataset(patches = val_patches,labels= val_labels, mean = mean, std = std)

        val_loader = DataLoader(
            dataset= val_dataset,
            shuffle= False,
            batch_size= 32,
            num_workers= 0,
            pin_memory= True
            )

        for epoch in range(num_epochs):
            
            
            train_loss,train_metrics = train_for_one_epoch(model, train_loader, device, aug, optimizer, criterion)
            val_loss, val_metrics = val_for_one_epoch(model, val_loader, metrics, device, criterion)

            # append valeus for epoch to history dict. Do i want an epoch key? 
            history_dict["run_ID"].append(run_ID)
            history_dict["loop"].append("outer")
            history_dict["epoch"].append(epoch+1)
            history_dict["train_loss"].append(train_loss)
            history_dict["val_loss"].append(val_loss)
            history_dict["val_accuracy"].append(val_metrics["accuracy"])
            history_dict["val_f1"].append(val_metrics["f1"])
            history_dict["val_precision"].append(val_metrics["precision"])
            history_dict["val_recall"].append(val_metrics["recall"])
            history_dict["val_auroc"].append(val_metrics["auroc"])
            history_dict["train_accuracy"].append(train_metrics["accuracy"])
            history_dict["train_f1"].append(train_metrics["f1"])
            history_dict["train_precision"].append(train_metrics["precision"])
            history_dict["train_recall"].append(train_metrics["recall"])
            history_dict["train_auroc"].append(train_metrics["auroc"])

            scheduler.step(val_loss)

            print(f"Epoch {epoch+1}/100 | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} "
                f"| val_f1: {val_metrics['f1']:.4f} | val_acc: {val_metrics['accuracy']:.4f}")

            # Early stopping

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_train_loss = train_loss
                best_val_metrics = val_metrics
                best_epoch = epoch
                epochs_no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve +=1
                if epochs_no_improve >= patience:
                    print(f"Early stopping at epcoch: {epoch+1}")
                    break

        # model.load_state_dict(best_state)

        loss_curve_fp = PROJECT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"
        plot_and_save_loss(history_dict, loss_curve_fp, run_ID)

        print(run_ID)
        print(f"Best epoch: {best_epoch+1} | val_loss: {best_val_loss:.4f} | val_metrics: {best_val_metrics}")

        
        list_of_run_histories.append(history_dict)

        inner_best_state_dict["run_ID"].append(run_ID)
        inner_best_state_dict["epoch_stopped"].append(best_epoch)
        inner_best_state_dict["test_region"].append(test_region)
        inner_best_state_dict["val_region"].append(val_region)
        inner_best_state_dict["train_loss"].append(train_loss)
        inner_best_state_dict["val_loss"].append(best_val_loss)
        inner_best_state_dict["val_accuracy"].append(best_val_metrics["accuracy"])
        inner_best_state_dict["val_f1"].append(best_val_metrics["f1"])
        inner_best_state_dict["val_precision"].append(best_val_metrics["precision"])
        inner_best_state_dict["val_recall"].append(best_val_metrics["recall"])
        inner_best_state_dict["val_auroc"].append(best_val_metrics["auroc"])

        list_of_inner_best_state_metrics.append(inner_best_state_dict)
    # -------------------------
    # Outer fold -> train on all remaining regions using median number of epochs from inner fold
    #---------------------------------------------------------------------------------------------
    history_dict = {"run_ID": [], "loop":[], "epoch":[], "train_loss": [], "train_accuracy":[],"train_f1":[], "train_precision":[], "train_recall":[], "train_auroc":[]
                        }
    
    test_mask = locations == test_region
    remaining_mask = locations != test_region

    run_ID = f"Test_{test_abrv}_{date}"

    remaining_patches = patches[remaining_mask] 
    remaining_mean = remaining_patches.mean(dim= (0,2,3))
    remaining_std =  remaining_patches.std(dim= (0,2,3))
    
    test_patches = patches[test_mask]
    
    remaining_labels = labels[remaining_mask]
    test_labels = labels[test_mask]
    test_label_ids = label_id[test_mask]
    test_class_int = class_int[test_mask]

    remaining_dataset = PatchDataset(patches= remaining_patches, labels=remaining_labels, mean = remaining_mean, std = remaining_std)
        
    remaining_loader = DataLoader(dataset= remaining_dataset,
            shuffle= True,
            batch_size= 32,
            num_workers= 0,
            pin_memory= True
            )
    
    test_dataset = PatchDataset(patches = test_patches, labels=test_labels, mean = remaining_mean, std = remaining_std)
    
    test_loader = DataLoader(dataset= test_dataset,
            shuffle= False,
            batch_size= 32,
            num_workers= 0,
            pin_memory= True
            )
    
    model = buildCNN_2xVGG(input_channels= input_channels, dropout = 0.3)
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    epochs_stopped = list(inner_best_state_dict["epoch_stopped"])
    median_epoch = int(median(epochs_stopped))

    for epoch in range(median_epoch):
        train_loss, train_metrics = train_for_one_epoch(model, remaining_loader, device, aug, optimizer, criterion)

        scheduler.step()

        print(f"Epoch {epoch+1}/100 | train_loss: {train_loss:.4f}")

        # save the per epoch metrics so can plot loss curve later
        history_dict["run_ID"].append(run_ID)
        history_dict["loop"].append("outer")
        history_dict["epoch"].append(epoch+1)
        history_dict["train_accuracy"].append(train_metrics["accuracy"])
        history_dict["train_f1"].append(train_metrics["f1"])
        history_dict["train_precision"].append(train_metrics["precision"])
        history_dict["train_recall"].append(train_metrics["recall"])
        history_dict["train_auroc"].append(train_metrics["auroc"])

        loss_curve_fp = PROJECT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"
        

        
    test_loss, test_metrics, preds_tensor, logits = predict_on_test_region(model, test_loader, metrics, device, criterion)

    loss_curve_fp = PROJECT_ROOT / "outputs/plots/loss_curves" / f"{run_ID}.png"
            
    plot_and_save_loss(history_dict, loss_curve_fp, run_ID)
    
    outer_metrics_dict["run_ID"].append(run_ID)
    outer_metrics_dict["test_region"].append(location)
    outer_metrics_dict["epochs_trained"].append(median_epoch)
    # Metrics for the prediction on held out test_region
    outer_metrics_dict["test_loss"].append(test_loss)
    outer_metrics_dict["test_accuracy"].append(test_metrics["accuracy"])
    outer_metrics_dict["test_f1"].append(test_metrics["f1"])
    outer_metrics_dict["test_precision"].append(test_metrics["precision"])
    outer_metrics_dict["test_recall"].append(test_metrics["recall"])
    outer_metrics_dict["test_auroc"].append(test_metrics["auroc"])

    # These are the train matrics from the final epoch
    outer_metrics_dict["train_loss"].append(train_loss)
    outer_metrics_dict["train_accuracy"].append(train_metrics["accuracy"])
    outer_metrics_dict["train_f1"].append(train_metrics["f1"])
    outer_metrics_dict["train_precision"].append(train_metrics["precision"])
    outer_metrics_dict["train_recall"].append(train_metrics["recall"])
    outer_metrics_dict["train_auroc"].append(train_metrics["auroc"])


# Need to put together the preds with labels etc
    preds_array = preds_tensor.detach().squeeze(1).cpu().numpy()
    labels_array = test_labels.detach().squeeze(1).numpy()

    prediction_df = pd.DataFrame({
        "labels": labels_array,
        "predicted": preds_array,
        "class_int": test_class_int,
        "test_region": test_region,
        "run_id": run_ID
    })

    list_of_predicition_dfs.append(prediction_df)

    # save trained models and the relevant parameters for each outer fold?

    checkpoint = {
        "run_ID": run_ID,
        "test_region": test_region,
        "model_state_dict": model.state_dict(),
        "mean": remaining_mean,
        "std": remaining_std,
        "band_list": band_list,
        "median_epoch": median_epoch,
        "hyperparams": {"lr": 1e-3, "weight_decay": 1e-4, "dropout": 0.3, "batch_size": 32},
    }
    model_fp = PROJECT_ROOT / "outputs" / "CNN_outputs" / "models" / f"{run_ID}.pt"
    model_fp.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, model_fp)
#----------------------------------------------------------------------------------
# After loops
# Concat the the prediciton_dfs
 
predictions = pd.concat(list_of_predicition_dfs,axis=0)

# Need to combine the history dicts into a data frame
history_df = pd.concat([pd.DataFrame(h) for h in list_of_run_histories], ignore_index= True)
    
run_history_fp = PROJECT_ROOT / "outputs" / "CNN_outputs" / f"training_run_history_{date}.csv"
history_df.to_csv(run_history_fp) 

# save the outer metrics as data frame
outer_metrics_df =pd.DataFrame(outer_metrics_dict)

outer_metrics_fp = PROJECT_ROOT / "outputs" / "CNN_outputs" / f"Outer_fold_metrics_{date}.csv"

outer_metrics_df.to_csv(outer_metrics_fp)
# save the best inner metrics as dataframe for each fold

inner_metrics_df = pd.concat([pd.DataFrame(h) for h in list_of_inner_best_state_metrics], ignore_index= True)

inner_metrics_fp = PROJECT_ROOT / "outputs" / "CNN_outputs" / f"Inner_fold_best_state_metrics_{date}.csv"
inner_metrics_df.to_csv(inner_metrics_fp)

